# 7.3 哪些位置應該學？

# 第 7 章：從續寫文字到回答問題

前置：第4章的因果模型與第1章的資料對齊。先用一筆不截斷對話，再回讀padding。完整render在tiny_perceptron/data.py；下列程式每節都新建資料，不依賴上一Notebook變量。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：兩種遮罩：看得見，不一定直接考它**

X 的 assistant 格預測 A；PAD 不當答案。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/masks.svg")))

**先想一想：**user不算loss，會從輸入中消失嗎？

我們常只要求模型學assistant答案，不要求它模仿user提問。忽略的位置設爲-100；這個數不是詞表ID。

**把直覺寫成數學：**有效監督mask=labels≠−100；attention mask另外決定能看到誰。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.data import render_chat

x, y = render_chat([{"role": "user", "content": "問"}, {"role": "assistant", "content": "答"}])
for i, (token, label) in enumerate(zip(x.tolist(), y.tolist())):
    print(i, token, label, "學" if label != -100 else "忽略直接loss")
assert (y != -100).sum() > 0

**如何讀結果：**位元組路線一箇中文答案有多個token；有效數不是字數。

**只改一件事：**只把答案改成兩個中文字。
